# Optional Lab A: Optimization & Grid Search for Motor Control**Computational Sensorimotor Control — Week 6 Supplement**This tutorial introduces the two optimization tools used in Week 6 and throughout the rest of the course:1. **scipy.optimize.minimize** — find the minimum of a cost function given constraints (used in static optimization, §6)2. **Grid search** — exhaustively test parameter combinations to find the best one (used in EPH parameter tuning, §1–2)Both are general-purpose tools. By the end of this tutorial you will understand when to use each, how to set them up, and how they connect to the motor control problems in the lecture.**No TODOs — this is a walkthrough.** Read each cell, run it, and make sure you understand the output before moving on.

In [ ]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
from matplotlib import cm
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})

---## Part 1: One-Dimensional OptimizationWe start with a simple cost function of one variable:$$J(x) = x \cdot e^{-x^2} + \frac{x^2}{20}$$The goal: find the value of x that minimizes J(x). This is the same problem structure as static optimization — we have a cost function and we want the inputs that make it smallest.

In [ ]:
# Define the cost function
def J_1d(x):
    return x * np.exp(-x**2) + (x**2) / 20

# Plot it
x = np.linspace(-4, 4, 200)
y = J_1d(x)

plt.figure(figsize=(8, 4))
plt.plot(x, y, '#2E86AB', lw=2)
plt.xlabel('x'); plt.ylabel('J(x)')
plt.title('1D cost function')
plt.axhline(0, color='gray', lw=0.5)
plt.show()

# Notice: there's a global minimum near x ≈ -0.7 and a local minimum near x ≈ 2

### Using scipy.optimize.minimize`sp.optimize.minimize(fun, x0)` takes:- `fun` — the function to minimize- `x0` — an initial guess (starting point for the search)It returns an object with `.x` (the optimal parameters) and `.fun` (the cost at the optimum).

In [ ]:
# Find the minimum starting from x0 = 0
result = sp.optimize.minimize(J_1d, x0=0.0)

print(result)
print(f'\nOptimal x = {result.x[0]:.4f}')
print(f'Minimum J = {result.fun:.4f}')

### The starting point matters`minimize` finds a **local** minimum — the nearest downhill point from where you start. Different starting points can lead to different solutions.

In [ ]:
# Try different starting points
for x0 in [-3, -1, 0, 1, 3]:
    res = sp.optimize.minimize(J_1d, x0=x0)
    print(f'Start x0={x0:+.0f}  →  optimal x={res.x[0]:+.4f},  J={res.fun:.4f}')

# Starting from x0=3 finds a different (worse) local minimum!

**Takeaway:** scipy.optimize.minimize is a *local* optimizer. It's fast and precise, but it only finds the minimum nearest to your starting point. For simple, convex problems (like static optimization with n ≥ 2), there's only one minimum and any starting point works. For non-convex problems, you may need to try multiple starting points or use a different approach — like grid search.

---## Part 2: Two-Dimensional OptimizationNow the cost function takes two inputs — like finding the best (dy, C) pair for EPH:$$J(x, y) = x \cdot e^{-(x^2 + y^2)} + \frac{x^2 + y^2}{20}$$

In [ ]:
def J_2d(X):
    x, y = X[0], X[1]
    return x * np.exp(-(x**2 + y**2)) + (x**2 + y**2) / 20

# Visualize the cost landscape
xv = np.linspace(-3, 3, 100)
yv = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(xv, yv)
Z = np.zeros_like(X)
for i in range(len(xv)):
    for j in range(len(yv)):
        Z[j, i] = J_2d([X[j, i], Y[j, i]])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.contourf(X, Y, Z, levels=30, cmap='viridis')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_title('Cost landscape (contour)')
ax1.plot(-0.67, 0, 'r*', ms=15, label='Global min'); ax1.legend()

ax2.plot_surface = None  # placeholder
im = ax2.contourf(X, Y, Z, levels=30, cmap='viridis')
plt.colorbar(im, ax=ax2, shrink=0.8)
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_title('Cost landscape (with colorbar)')
plt.tight_layout(); plt.show()

In [ ]:
# Minimize from different starting points
for x0 in [[-2, -2], [0, 0], [2, 2], [-1, 1]]:
    res = sp.optimize.minimize(J_2d, x0)
    print(f'Start ({x0[0]:+.0f},{x0[1]:+.0f})  →  optimum ({res.x[0]:+.4f},{res.x[1]:+.4f}),  J={res.fun:.4f}')

### Adding constraintsIn static optimization (Lecture §6), we need τ = R·F *and* F ≥ 0. scipy handles this with the `constraints` and `bounds` arguments.Let's see a constrained example: minimize J(x,y) subject to x + y = 1.

In [ ]:
# Constrained optimization: minimize J(x,y) subject to x + y = 1
constraint = {'type': 'eq', 'fun': lambda X: X[0] + X[1] - 1}

res_free = sp.optimize.minimize(J_2d, [0, 0])
res_con  = sp.optimize.minimize(J_2d, [0, 0], constraints=constraint)

print(f'Free optimum:        ({res_free.x[0]:+.4f}, {res_free.x[1]:+.4f}), J = {res_free.fun:.4f}')
print(f'Constrained (x+y=1): ({res_con.x[0]:+.4f}, {res_con.x[1]:+.4f}), J = {res_con.fun:.4f}')

# The constraint forces the solution onto the line x+y=1

In [ ]:
# Adding bounds: x ≥ 0 and y ≥ 0 (like F ≥ 0 for muscle forces)
bounds = [(0, None), (0, None)]

res_bounded = sp.optimize.minimize(J_2d, [1, 1], bounds=bounds, constraints=constraint)
print(f'Bounded + constrained: ({res_bounded.x[0]:.4f}, {res_bounded.x[1]:.4f}), J = {res_bounded.fun:.4f}')
print('Both x and y are non-negative, and x + y = 1.')

**Connection to static optimization:** In Lecture §6, the cost function is Σ(Fᵢ/ρᵢ)ⁿ, the constraint is R·F = τ (an equality), and the bounds are F ≥ 0 (non-negativity). The call looks like:```pythonsp.optimize.minimize(cost, x0, method='SLSQP', bounds=bounds, constraints=constraints)```SLSQP (Sequential Least Squares Programming) is the method that handles both equality constraints and bounds simultaneously.

---## Part 3: Grid Search — When scipy Isn't Enoughscipy.optimize.minimize is powerful but requires:1. A differentiable cost function2. A good starting point3. The problem to be reasonably smoothFor EPH parameter tuning (Lecture §1–2), none of these hold. The "cost function" is: run a full muscle simulation → measure path deviation. This is a black-box simulation — not differentiable, possibly noisy, with an unknown landscape. The solution: **grid search** — try every combination of parameters on a grid and pick the best one.**Important note for neuroscience:** The brain almost certainly does NOT run a grid search for every new task. Through repeated practice, the nervous system learns appropriate parameters for familiar conditions — effectively building a lookup table from experience. We use grid search here as a computational tool to find EPH's theoretical best performance, not as a model of what the brain does.

In [ ]:
# ── Simple grid search example ──
# Problem: find (a, b) that minimize f(a, b) = (a - 1.7)^2 + (b - 3.2)^2

def f_simple(a, b):
    return (a - 1.7)**2 + (b - 3.2)**2

# Define the grid
a_values = np.linspace(0, 5, 50)
b_values = np.linspace(0, 5, 50)

# Search
best_cost = np.inf
best_a, best_b = None, None

for a in a_values:
    for b in b_values:
        cost = f_simple(a, b)
        if cost < best_cost:
            best_cost = cost
            best_a, best_b = a, b

print(f'Grid search result: a = {best_a:.2f}, b = {best_b:.2f}, cost = {best_cost:.4f}')
print(f'True optimum:       a = 1.70, b = 3.20, cost = 0.0000')
print(f'Grid resolution:    {a_values[1]-a_values[0]:.3f} per step')
print(f'Total evaluations:  {len(a_values) * len(b_values)}')

### Grid search tradeoffsThe grid resolution determines the accuracy. Finer grids find better solutions but take longer.

In [ ]:
# Compare grid resolutions
for n_points in [5, 10, 25, 50, 100]:
    a_vals = np.linspace(0, 5, n_points)
    b_vals = np.linspace(0, 5, n_points)
    best = np.inf; ba = bb = 0
    for a in a_vals:
        for b in b_vals:
            c = f_simple(a, b)
            if c < best: best = c; ba = a; bb = b
    error = np.sqrt((ba - 1.7)**2 + (bb - 3.2)**2)
    print(f'Grid {n_points:3d}x{n_points:3d} ({n_points**2:5d} evals): '
          f'found ({ba:.2f}, {bb:.2f}), error = {error:.3f}')

### Grid search vs scipy| | scipy.optimize.minimize | Grid search ||---|---|---|| **Speed** | Fast (gradient-based) | Slow (exhaustive) || **Accuracy** | High (converges to local optimum) | Limited by grid resolution || **Requirements** | Differentiable cost function | Any black-box function || **Global vs local** | Local (depends on starting point) | Global (tests everything) || **Typical use** | Static optimization (§6) | EPH parameter tuning (§1–2) |In Week 6, we use **both**: grid search to find EPH's best (dy, C) parameters (because the simulation is a black box), and scipy to solve the static optimization at each timestep (because the Crowninshield cost is smooth and convex).

---## Part 4: Applying Both Tools to Motor ControlNow we connect these tools to the Week 6 problems.

In [ ]:
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
from smc import (
    Arm, make_muscles, simulate_lambda,
    Q_REF, MUSCLE_DEFS,
    mass_matrix, coriolis, joint_accelerations, rk4_step,
)

arm = Arm()
ik  = arm.inverse_kinematics
fk  = arm.forward_kinematics
start_hand = fk(Q_REF)

R_ARC = 0.06; T_MOVE = 0.800
F_LOAD = np.array([0.0, -3.0])
J_ref = arm.jacobian(Q_REF)
tau_ext = J_ref.T @ F_LOAD
perturb_3N = lambda t: tau_ext

def lambda_for_posture(q_target, C=0.25):
    muscles = make_muscles()
    lam = np.zeros(6)
    for j, m in enumerate(muscles):
        l_eq = m.length(q_target)
        shift = (abs(m.r_sh) + abs(m.r_el)) * C
        lam[j] = l_eq - shift
    return lam

def dense_arc(R, n=500):
    center = start_hand + np.array([R, 0])
    th = np.linspace(np.pi, 0, n)
    return np.column_stack([center[0]+R*np.cos(th), center[1]+R*np.sin(th)])

def max_path_deviation(hand, desired):
    return max(np.linalg.norm(desired - hand[i], axis=1).min() for i in range(len(hand)))

print('smc loaded. Ready for motor control examples.')

### Application 1: Grid search for EPH R-shift and C-commandThis is exactly what Lecture §1–2 does: for each (dy, C) pair, run a full EPH simulation on the 6 cm arc under 3 N load, measure path deviation, and keep the best.

In [ ]:
def eph_arc(R, dy=0, C=0.25, perturb_fn=None):
    _, tip, end = start_hand + np.array([R, R]), start_hand + np.array([R, R]), start_hand + np.array([2*R, 0])
    tip_s = start_hand + np.array([R, R]) + np.array([0, dy])
    end_s = start_hand + np.array([2*R, 0]) + np.array([0, dy])
    try:
        q_tip = ik(tip_s[0], tip_s[1]); q_end = ik(end_s[0], end_s[1])
        if not arm.in_joint_limits(q_tip) or not arm.in_joint_limits(q_end): return None
    except: return None
    li = lambda_for_posture(Q_REF, C)
    l_tip = lambda_for_posture(q_tip, C); l_end = lambda_for_posture(q_end, C)
    ramp = T_MOVE / 2
    def lam_fn(t):
        if t < ramp: return li + (t/ramp)*(l_tip - li)
        elif t < 2*ramp: return l_tip + ((t-ramp)/ramp)*(l_end - l_tip)
        else: return l_end.copy()
    _, _, hand, _ = simulate_lambda(lam_fn, T=1.2, dt=0.0005, perturb_fn=perturb_fn)
    return hand

des6 = dense_arc(R_ARC)

# Grid search
dy_values = np.linspace(0, 0.08, 12)
C_values  = np.linspace(0.20, 0.60, 10)

best_pd = np.inf; best_dy = 0; best_C = 0.25
cost_landscape = np.full((len(C_values), len(dy_values)), np.nan)

print(f'Grid: {len(dy_values)} x {len(C_values)} = {len(dy_values)*len(C_values)} simulations')

for i, dy in enumerate(dy_values):
    for j, C in enumerate(C_values):
        h = eph_arc(R_ARC, dy=dy, C=C, perturb_fn=perturb_3N)
        if h is None: continue
        pd = max_path_deviation(h, des6) * 100
        cost_landscape[j, i] = pd
        if pd < best_pd:
            best_pd = pd; best_dy = dy; best_C = C

print(f'\nBest: dy = {best_dy*100:.1f} cm, C = {best_C:.2f} rad, path dev = {best_pd:.2f} cm')

In [ ]:
# Visualize the cost landscape
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.contourf(dy_values*100, C_values, cost_landscape, levels=20, cmap='viridis')
ax.plot(best_dy*100, best_C, 'r*', ms=15, label=f'Best: ({best_dy*100:.1f}, {best_C:.2f})')
plt.colorbar(im, ax=ax, label='Path deviation (cm)')
ax.set_xlabel('R-shift dy (cm)'); ax.set_ylabel('C-command (rad)')
ax.set_title('EPH grid search: cost landscape under 3 N load')
ax.legend()
plt.tight_layout(); plt.show()

print('The landscape shows that the optimal region is a narrow valley —')
print('small changes in dy or C can significantly worsen performance.')
print('This is why the brain would need to learn these parameters precisely.')

### Application 2: Static optimization with scipyThis is the constrained optimization from Lecture §6: distribute a required joint torque among 6 muscles by minimizing the Crowninshield cost.

In [ ]:
# Moment arm matrix and force scaling
R_mat = np.array([[m[3] for m in MUSCLE_DEFS], [m[4] for m in MUSCLE_DEFS]])
rho = np.array([m[1] for m in MUSCLE_DEFS])
mnames = ['Pec', 'BicL', 'BicS', 'Delt', 'TriL', 'TriLg']

# A sample required torque (from the 6 cm arc under load, at peak)
tau_required = np.array([-2.44, -0.94])  # shoulder, elbow (N·m)

def crowninshield_cost(F, n_power=3):
    return np.sum((F / rho) ** n_power)

def crowninshield_jac(F, n_power=3):
    return n_power * (F / rho) ** (n_power - 1) / rho

# Constraints: R · F = tau (equality)
constraints = {
    'type': 'eq',
    'fun': lambda F: R_mat @ F - tau_required,
    'jac': lambda F: R_mat
}

# Bounds: F ≥ 0 (muscles can only pull, not push)
bounds = [(0, None)] * 6

# Solve
result = sp.optimize.minimize(
    crowninshield_cost,
    x0=np.ones(6) * 0.1,       # initial guess
    jac=crowninshield_jac,       # gradient (speeds up convergence)
    method='SLSQP',             # handles equality constraints + bounds
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 500, 'ftol': 1e-12}
)

print(f'Success: {result.success}')
print(f'Optimal forces:')
for j in range(6):
    print(f'  {mnames[j]:5s}: {result.x[j]:6.1f} N')
print(f'\nVerification: R @ F = ({R_mat @ result.x})')
print(f'Required tau:       ({tau_required})')
print(f'Match: {np.allclose(R_mat @ result.x, tau_required, atol=1e-6)}')

### Why scipy works here but not for EPHThe Crowninshield cost is **smooth and convex** (for n ≥ 2): scipy can compute gradients, follow them downhill, and guarantee convergence to the unique global minimum. The constraints (R·F = τ) are linear, and the bounds (F ≥ 0) are simple box constraints. SLSQP handles all of this efficiently.EPH parameter tuning is **none of these things**: the "cost function" (path deviation after a full simulation) is a black box that takes 0.5 seconds per evaluation, is not differentiable, may be noisy, and has an unknown landscape with possible local minima. Grid search is the right tool for this problem.**In the lecture and lab, we use both:**- Grid search → find (dy, C) for EPH (black-box simulation, ~120 evaluations)- scipy.optimize.minimize → solve τ = R·F at each timestep (smooth, convex, ~800 calls per trajectory)

---## Summary| Tool | When to use | Week 6 application ||---|---|---|| `scipy.optimize.minimize` | Smooth, differentiable cost; need constraints and bounds | Static optimization (§6): minimize Σ(Fᵢ/ρᵢ)ⁿ s.t. R·F = τ, F ≥ 0 || Grid search | Black-box function; few parameters (2–3); need global search | EPH tuning (§1–2): find best (dy, C) by running simulations |**Key points:**- scipy is fast and precise but finds local minima — the cost must be convex or you need multiple starting points- Grid search is slow but global — it tests everything, limited only by resolution and the number of parameters- The brain probably uses neither: it learns from experience, building internal models that generalize to new conditions. That's the subject of Weeks 8–11.